# MLflow Experiment Tracking — Covered Call Strategy Classification

This notebook integrates MLflow tracking across all trained models in this project:
- **XGBoost baseline** (`06_XGBoost_baseline`)
- **PatchTST** (`07_PatchTSTfor_Classification`)
- **LSTM-CNN + Attention** (`09_LSTM_CNN_Classification`) — original, regularised

**What is tracked per run:**
- Hyperparameters (Optuna best params)
- Metrics: Accuracy, Macro-F1, Balanced Accuracy (val + test)
- Overfitting gap (train_F1 - val_F1)
- Confusion matrix, ROC curve plots
- Model artifacts (.pth files)
- Feature importance / architecture summary

**Run MLflow UI:**
```bash
mlflow ui --backend-store-uri ./mlruns
# Open http://127.0.0.1:5000
```

## 1. Imports & MLflow Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import json
import tempfile
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler, label_binarize
from sklearn.metrics import (
    f1_score, accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix,
    roc_curve, auc, average_precision_score
)

import mlflow
import mlflow.pytorch

# ── Paths ──────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('../')
DATA_PATH    = PROJECT_ROOT / 'data/clean/daily_stock_optimal_bucket_modeling_with_fred.parquet'
MODEL_DIR       = PROJECT_ROOT / 'saved_models'
ARTIFACTS_DIR   = PROJECT_ROOT / 'saved_artifacts'
MLRUNS_DIR   = PROJECT_ROOT / 'mlruns'

# ── MLflow experiment ──────────────────────────────────────────────────────
mlflow.set_tracking_uri(str(MLRUNS_DIR))
EXPERIMENT_NAME = 'covered-call-strategy-classification'
mlflow.set_experiment(EXPERIMENT_NAME)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('MLflow version   :', mlflow.__version__)
print('Tracking URI     :', mlflow.get_tracking_uri())
print('Experiment       :', EXPERIMENT_NAME)
print('Device           :', device)

## 2. Shared Data Preparation
Load data, build sequences and splits — same logic as notebook 09.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset, DataLoader

SEQ_LEN        = 50
N_TOP_FEATURES = 35
RANDOM_STATE   = 42

# Load & remap
df = pd.read_parquet(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)
bucket_map = {'OTM10_60':'OTM10_60_90','OTM10_90':'OTM10_60_90',
              'OTM5_60':'OTM5_60_90','OTM5_90':'OTM5_60_90'}
df['optimal_bucket'] = df['optimal_bucket'].replace(bucket_map)
target_encoder = LabelEncoder()
df['target']   = target_encoder.fit_transform(df['optimal_bucket'])
num_classes    = len(target_encoder.classes_)
class_names    = list(target_encoder.classes_)

# Temporal split
train_df = df[df['date'] <  '2022-01-01'].copy()
val_df   = df[(df['date'] >= '2022-01-01') & (df['date'] < '2024-01-01')].copy()
test_df  = df[df['date'] >= '2024-01-01'].copy()

# Feature selection
exclude_cols = ['symbol','date','fiscalDateEnding','optimal_bucket','target']
all_feat     = [c for c in train_df.columns if c not in exclude_cols]
X_rf = train_df[all_feat].select_dtypes(include=['number']).fillna(0)
rf   = RandomForestClassifier(n_estimators=300, max_depth=12,
                               class_weight='balanced_subsample',
                               random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_rf, train_df['target'])
feature_cols = pd.Series(rf.feature_importances_, index=X_rf.columns).nlargest(N_TOP_FEATURES).index.tolist()

# Scale
scaler = StandardScaler()
for split in [train_df, val_df, test_df]:
    split[feature_cols] = split[feature_cols].fillna(0)
scaler.fit(train_df[feature_cols])
train_df[feature_cols] = scaler.transform(train_df[feature_cols])
val_df[feature_cols]   = scaler.transform(val_df[feature_cols])
test_df[feature_cols]  = scaler.transform(test_df[feature_cols])

# Sequences
def build_sequences(panel_df, feature_cols, seq_len=50):
    X_seq, y_seq = [], []
    for sym, grp in panel_df.groupby('symbol'):
        grp = grp.sort_values('date').reset_index(drop=True)
        if len(grp) < seq_len: continue
        X_vals, y_vals = grp[feature_cols].values, grp['target'].values
        for i in range(seq_len-1, len(grp)):
            X_seq.append(X_vals[i-seq_len+1:i+1])
            y_seq.append(y_vals[i])
    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.int64)

X_train_seq, y_train_seq = build_sequences(train_df, feature_cols)
X_val_seq,   y_val_seq   = build_sequences(val_df,   feature_cols)
X_test_seq,  y_test_seq  = build_sequences(test_df,  feature_cols)

print(f'Train: {X_train_seq.shape} | Val: {X_val_seq.shape} | Test: {X_test_seq.shape}')
print(f'Classes: {class_names}')

## 3. Shared Utilities

In [ ]:
class CoveredCallDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return {'past_values': self.X[idx], 'target': self.y[idx]}

def make_loader(X, y, batch_size, shuffle=True):
    return DataLoader(CoveredCallDataset(X, y), batch_size=batch_size,
                      shuffle=shuffle, num_workers=0)

@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval()
    all_preds, all_targets, all_probs = [], [], []
    for batch in loader:
        x = batch['past_values'].to(device)
        y = batch['target']
        logits = model(x)
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()
        all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
        all_targets.extend(y.numpy())
        all_probs.extend(probs)
    return np.array(all_targets), np.array(all_preds), np.array(all_probs)

def compute_metrics(y_true, y_pred):
    return {
        'accuracy'         : round(accuracy_score(y_true, y_pred), 4),
        'macro_f1'         : round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'balanced_accuracy': round(balanced_accuracy_score(y_true, y_pred), 4),
    }

def plot_confusion_matrix_fig(y_true, y_pred, labels, title):
    cm  = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title(title)
    plt.tight_layout()
    return fig

def plot_roc_fig(y_true, probs, n_classes, class_names, title):
    y_bin = label_binarize(y_true, classes=list(range(n_classes)))
    fig, ax = plt.subplots(figsize=(9, 7))
    for i, name in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
        ax.plot(fpr, tpr, label=f'{name} (AUC={auc(fpr,tpr):.2f})')
    ax.plot([0,1],[0,1],'k--'); ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(title); ax.legend(fontsize=8, loc='lower right')
    plt.tight_layout()
    return fig

print('Utilities ready.')

## 4. MLflow Logging Helper

In [ ]:
def log_run_to_mlflow(
    run_name,
    model_type,
    params,
    val_metrics,
    test_metrics,
    train_history=None,
    model=None,
    model_path=None,
    val_preds=None,
    test_preds=None,
    val_probs=None,
    test_probs=None,
    tags=None,
):
    """
    Log a complete model run to MLflow.
    Logs: params, val/test metrics, training curves, confusion matrix,
          ROC curves, classification report, and model artifact.
    """
    with mlflow.start_run(run_name=run_name) as run:

        # ── Tags ────────────────────────────────────────────────────────
        mlflow.set_tag('model_type', model_type)
        mlflow.set_tag('dataset', 'daily_stock_optimal_bucket_modeling_with_fred')
        mlflow.set_tag('seq_len', SEQ_LEN)
        mlflow.set_tag('n_features', N_TOP_FEATURES)
        mlflow.set_tag('n_classes', num_classes)
        if tags:
            for k, v in tags.items():
                mlflow.set_tag(k, v)

        # ── Hyperparameters ─────────────────────────────────────────────
        mlflow.log_params(params)

        # ── Val metrics (prefixed val_) ──────────────────────────────────
        for k, v in val_metrics.items():
            mlflow.log_metric(f'val_{k}', v)

        # ── Test metrics (prefixed test_) ────────────────────────────────
        for k, v in test_metrics.items():
            mlflow.log_metric(f'test_{k}', v)

        # ── Overfitting gap ─────────────────────────────────────────────
        if train_history:
            gap = train_history['train_f1'][-1] - train_history['val_f1'][-1]
            mlflow.log_metric('overfit_gap', round(gap, 4))

            # Training curves as artifact
            epochs = range(1, len(train_history['train_loss']) + 1)
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            axes[0].plot(epochs, train_history['train_loss'], label='Train')
            axes[0].plot(epochs, train_history['val_loss'],   label='Val')
            axes[0].set_title('Loss'); axes[0].legend()
            axes[1].plot(epochs, train_history['train_f1'],  label='Train')
            axes[1].plot(epochs, train_history['val_f1'],    label='Val')
            axes[1].set_title('Macro-F1'); axes[1].legend()
            plt.suptitle(f'{run_name} — Training History')
            plt.tight_layout()
            with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
                plt.savefig(f.name, dpi=100)
                mlflow.log_artifact(f.name, artifact_path='plots/training_curves')
            plt.close()

        # ── Confusion matrix ─────────────────────────────────────────────
        if test_preds is not None:
            fig = plot_confusion_matrix_fig(
                test_preds[0], test_preds[1], class_names,
                f'{run_name} — Confusion Matrix (Test)'
            )
            with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
                fig.savefig(f.name, dpi=100)
                mlflow.log_artifact(f.name, artifact_path='plots/confusion_matrix')
            plt.close(fig)

        # ── ROC curves ───────────────────────────────────────────────────
        if test_probs is not None:
            fig = plot_roc_fig(
                test_preds[0], test_probs, num_classes, class_names,
                f'{run_name} — ROC Curves (Test)'
            )
            with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
                fig.savefig(f.name, dpi=100)
                mlflow.log_artifact(f.name, artifact_path='plots/roc_curves')
            plt.close(fig)

        # ── Classification report as text artifact ───────────────────────
        if test_preds is not None:
            report = classification_report(
                test_preds[0], test_preds[1],
                target_names=class_names, zero_division=0
            )
            with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
                f.write(f'Run: {run_name}\n\n')
                f.write(report)
                mlflow.log_artifact(f.name, artifact_path='reports')

        # ── Model artifact (.pth file) ────────────────────────────────────
        if model_path and Path(model_path).exists():
            mlflow.log_artifact(str(model_path), artifact_path='model_checkpoints')

        print(f'Run logged: {run_name}')
        print(f'  Run ID : {run.info.run_id}')
        print(f'  Val  Macro-F1  : {val_metrics.get("macro_f1", "N/A")}')
        print(f'  Test Macro-F1  : {test_metrics.get("macro_f1", "N/A")}')
        return run.info.run_id

print('log_run_to_mlflow() ready.')

## 5. Load & Register LSTM-CNN Models

In [ ]:
# ── Rebuild model class (needed to load state dicts) ───────────────────────
class MultiScaleCNN(nn.Module):
    def __init__(self, n_features, cnn_out_channels, dropout):
        super().__init__()
        self.branches = nn.ModuleList([
            self._make_branch(n_features, cnn_out_channels, k, dropout)
            for k in [3, 5, 7]
        ])
    @staticmethod
    def _make_branch(in_ch, out_ch, kernel, dropout):
        pad = kernel // 2
        return nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=kernel, padding=pad),
            nn.BatchNorm1d(out_ch), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(out_ch, out_ch, kernel_size=kernel, padding=pad),
            nn.BatchNorm1d(out_ch), nn.ReLU(), nn.Dropout(dropout),
            nn.AdaptiveAvgPool1d(1),
        )
    def forward(self, x):
        return torch.cat([b(x).squeeze(-1) for b in self.branches], dim=-1)

class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim, attn_dim=64):
        super().__init__()
        self.W = nn.Linear(hidden_dim, attn_dim, bias=True)
        self.v = nn.Linear(attn_dim, 1, bias=False)
    def forward(self, lstm_out):
        weight = torch.softmax(self.v(torch.tanh(self.W(lstm_out))), dim=1)
        return (weight * lstm_out).sum(dim=1), weight.squeeze(-1)

class LSTMCNNClassifier(nn.Module):
    def __init__(self, n_features, seq_len, num_classes,
                 cnn_out_channels=64, lstm_hidden=128,
                 lstm_layers=2, attn_dim=64, dropout=0.2):
        super().__init__()
        self.multiscale_cnn = MultiScaleCNN(n_features, cnn_out_channels, dropout)
        self.lstm = nn.LSTM(n_features, lstm_hidden, lstm_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if lstm_layers > 1 else 0.0)
        self.attention = TemporalAttention(lstm_hidden * 2, attn_dim)
        fusion_dim = cnn_out_channels * 3 + lstm_hidden * 2
        self.head = nn.Sequential(
            nn.LayerNorm(fusion_dim), nn.Dropout(dropout),
            nn.Linear(fusion_dim, fusion_dim // 2), nn.ReLU(),
            nn.Dropout(dropout / 2), nn.Linear(fusion_dim // 2, num_classes),
        )
    def forward(self, past_values):
        x_cnn  = self.multiscale_cnn(past_values.transpose(1, 2))
        lstm_out, _ = self.lstm(past_values)
        x_attn, _   = self.attention(lstm_out)
        return self.head(torch.cat([x_cnn, x_attn], dim=-1))


def load_lstm_cnn(model_path, device):
    ckpt = torch.load(model_path, map_location=device)
    p    = ckpt['best_params']
    model = LSTMCNNClassifier(
        n_features       = len(ckpt['feature_cols']),
        seq_len          = ckpt['seq_len'],
        num_classes      = ckpt['num_classes'],
        cnn_out_channels = p.get('cnn_out_channels', 64),
        lstm_hidden      = p.get('lstm_hidden', 128),
        lstm_layers      = p.get('lstm_layers', 2),
        attn_dim         = p.get('attn_dim', 64),
        dropout          = p.get('dropout', 0.2),
    ).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    return model, ckpt

print('Model classes and loader ready.')

## 6. Log LSTM-CNN Best Model

In [ ]:
MODEL_PATH_BEST = MODEL_DIR / 'lstm_cnn_best_model.pth'

model_best, ckpt_best = load_lstm_cnn(MODEL_PATH_BEST, device)

val_loader  = make_loader(X_val_seq,  y_val_seq,  128, shuffle=False)
test_loader = make_loader(X_test_seq, y_test_seq, 128, shuffle=False)

y_val_true,  y_val_pred,  y_val_probs  = get_predictions(model_best, val_loader,  device)
y_test_true, y_test_pred, y_test_probs = get_predictions(model_best, test_loader, device)

val_metrics  = compute_metrics(y_val_true,  y_val_pred)
test_metrics = compute_metrics(y_test_true, y_test_pred)

run_id_best = log_run_to_mlflow(
    run_name     = 'LSTM-CNN-Best',
    model_type   = 'MultiScale-CNN + BiLSTM + Attention',
    params       = ckpt_best['best_params'],
    val_metrics  = val_metrics,
    test_metrics = test_metrics,
    model_path   = MODEL_PATH_BEST,
    test_preds   = (y_test_true, y_test_pred),
    test_probs   = y_test_probs,
    tags         = {'variant': 'best_optuna', 'regularised': 'false'},
)
print(f'\nVal  metrics : {val_metrics}')
print(f'Test metrics : {test_metrics}')

## 7. Log LSTM-CNN Regularised Model

In [ ]:
MODEL_PATH_REG = MODEL_DIR / 'lstm_cnn_regularised_model.pth'

if MODEL_PATH_REG.exists():
    model_reg, ckpt_reg = load_lstm_cnn(MODEL_PATH_REG, device)

    y_val_true_r,  y_val_pred_r,  y_val_probs_r  = get_predictions(model_reg, val_loader,  device)
    y_test_true_r, y_test_pred_r, y_test_probs_r = get_predictions(model_reg, test_loader, device)

    run_id_reg = log_run_to_mlflow(
        run_name     = 'LSTM-CNN-Regularised',
        model_type   = 'MultiScale-CNN + BiLSTM + Attention',
        params       = ckpt_reg['best_params'],
        val_metrics  = compute_metrics(y_val_true_r,  y_val_pred_r),
        test_metrics = compute_metrics(y_test_true_r, y_test_pred_r),
        model_path   = MODEL_PATH_REG,
        test_preds   = (y_test_true_r, y_test_pred_r),
        test_probs   = y_test_probs_r,
        tags         = {'variant': 'regularised', 'regularised': 'true',
                        'dropout': str(ckpt_reg['best_params'].get('dropout')),
                        'weight_decay': str(ckpt_reg['best_params'].get('weight_decay'))},
    )
else:
    print('Regularised model not found — skipping. Run Section 13b in notebook 09 first.')

## 8. Log XGBoost Baseline Model

In [ ]:
import xgboost as xgb
import joblib

ARTIFACTS_DIR   = PROJECT_ROOT / 'saved_artifacts'
XGB_MODEL_PATH  = ARTIFACTS_DIR / 'models/xgboost_model_s.json'
XGB_PARAMS_PATH = ARTIFACTS_DIR / 'params/best_params.json'
XGB_FEAT_PATH   = ARTIFACTS_DIR / 'features/feature_columns.pkl'
MODEL_DATA_PATH = PROJECT_ROOT / 'model_datasets'

# Load model and metadata
xgb_model   = xgb.XGBClassifier()
xgb_model.load_model(XGB_MODEL_PATH)
with open(XGB_PARAMS_PATH) as f:
    xgb_params = json.load(f)
xgb_features = joblib.load(XGB_FEAT_PATH)

# Load flat (non-sequence) val/test sets from model_datasets
X_val_flat  = pd.read_parquet(MODEL_DATA_PATH / 'X_val_scaled.parquet')[xgb_features]
X_test_flat = pd.read_parquet(MODEL_DATA_PATH / 'X_test_scaled.parquet')[xgb_features]
y_val_flat  = pd.read_parquet(MODEL_DATA_PATH / 'y_val.parquet').squeeze()
y_test_flat = pd.read_parquet(MODEL_DATA_PATH / 'y_test.parquet').squeeze()

xgb_val_pred  = xgb_model.predict(X_val_flat)
xgb_test_pred = xgb_model.predict(X_test_flat)
xgb_test_probs = xgb_model.predict_proba(X_test_flat)

xgb_val_metrics  = compute_metrics(y_val_flat,  xgb_val_pred)
xgb_test_metrics = compute_metrics(y_test_flat, xgb_test_pred)

# Remove non-loggable keys from params
log_params = {k: v for k, v in xgb_params.items()
              if isinstance(v, (int, float, str, bool))}

run_id_xgb = log_run_to_mlflow(
    run_name     = 'XGBoost-Baseline',
    model_type   = 'XGBoost',
    params       = log_params,
    val_metrics  = xgb_val_metrics,
    test_metrics = xgb_test_metrics,
    model_path   = XGB_MODEL_PATH,
    test_preds   = (y_test_flat.values, xgb_test_pred),
    test_probs   = xgb_test_probs,
    tags         = {'variant': 'baseline', 'framework': 'xgboost'},
)
print(f'\nVal  metrics : {xgb_val_metrics}')
print(f'Test metrics : {xgb_test_metrics}')

## 9. Log PatchTST Transformer Model

In [ ]:
from transformers import PatchTSTConfig, PatchTSTForClassification

PATCHTST_PATH = ARTIFACTS_DIR / 'models/patchtst_best_model_with_fred.pth'

ckpt_ptst = torch.load(PATCHTST_PATH, map_location=device)
p_ptst    = ckpt_ptst['best_params']

# Rebuild PatchTST model
config_ptst = PatchTSTConfig(
    num_input_channels  = len(ckpt_ptst['feature_cols']),
    context_length      = ckpt_ptst['seq_len'],
    patch_length        = p_ptst['patch_length'],
    stride              = p_ptst['stride'],
    num_targets         = ckpt_ptst['num_classes'],
    d_model             = p_ptst['d_model'],
    num_attention_heads = p_ptst['num_attention_heads'],
    num_hidden_layers   = p_ptst['num_hidden_layers'],
    ffn_dim             = p_ptst['ffn_dim'],
    dropout             = p_ptst['dropout'],
    head_dropout        = p_ptst['head_dropout'],
    norm_type           = 'batchnorm',
)
ptst_model = PatchTSTForClassification(config_ptst).to(device)
ptst_model.load_state_dict(ckpt_ptst['model_state_dict'])
ptst_model.eval()

# PatchTST uses sequences of the features it was trained on
ptst_feat_cols = ckpt_ptst['feature_cols']

# Rebuild val/test sequences using PatchTST feature set
X_val_ptst,  y_val_ptst  = build_sequences(val_df,  ptst_feat_cols, seq_len=ckpt_ptst['seq_len'])
X_test_ptst, y_test_ptst = build_sequences(test_df, ptst_feat_cols, seq_len=ckpt_ptst['seq_len'])

# Inference
@torch.no_grad()
def predict_patchtst(model, X, batch_size=128, device=device):
    all_preds, all_probs, all_targets = [], [], []
    # PatchTST expects (batch, seq_len, n_features)
    for start in range(0, len(X), batch_size):
        x_batch = torch.tensor(X[start:start+batch_size], dtype=torch.float32).to(device)
        out     = model(past_values=x_batch)
        logits  = out.prediction_logits
        probs   = torch.softmax(logits, dim=-1).cpu().numpy()
        all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
        all_probs.extend(probs)
    return np.array(all_preds), np.array(all_probs)

ptst_val_pred,  ptst_val_probs  = predict_patchtst(ptst_model, X_val_ptst)
ptst_test_pred, ptst_test_probs = predict_patchtst(ptst_model, X_test_ptst)

ptst_val_metrics  = compute_metrics(y_val_ptst,  ptst_val_pred)
ptst_test_metrics = compute_metrics(y_test_ptst, ptst_test_pred)

run_id_ptst = log_run_to_mlflow(
    run_name     = 'PatchTST-Transformer',
    model_type   = 'PatchTST (HuggingFace Transformer)',
    params       = p_ptst,
    val_metrics  = ptst_val_metrics,
    test_metrics = ptst_test_metrics,
    model_path   = PATCHTST_PATH,
    test_preds   = (y_test_ptst, ptst_test_pred),
    test_probs   = ptst_test_probs,
    tags         = {'variant': 'patchtst_optuna', 'framework': 'huggingface'},
)
print(f'\nVal  metrics : {ptst_val_metrics}')
print(f'Test metrics : {ptst_test_metrics}')

## 10. Compare All Runs

In [ ]:
# Pull all runs from the experiment
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs_df    = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=['metrics.test_macro_f1 DESC']
)

# Select key columns
display_cols = ['tags.mlflow.runName', 'tags.model_type', 'tags.variant',
                'metrics.val_macro_f1', 'metrics.test_macro_f1',
                'metrics.val_balanced_accuracy', 'metrics.test_balanced_accuracy',
                'metrics.overfit_gap']
available   = [c for c in display_cols if c in runs_df.columns]
summary     = runs_df[available].rename(columns=lambda c: c.replace('tags.','').replace('metrics.',''))

print('=== All MLflow Runs — Ranked by Test Macro-F1 ===')
display(summary.reset_index(drop=True))

In [ ]:
# Bar chart comparison
if 'metrics.test_macro_f1' in runs_df.columns and 'metrics.val_macro_f1' in runs_df.columns:
    plot_df = runs_df[['tags.mlflow.runName', 'metrics.val_macro_f1',
                        'metrics.test_macro_f1']].dropna()
    plot_df = plot_df.rename(columns={'tags.mlflow.runName': 'Run',
                                       'metrics.val_macro_f1': 'Val Macro-F1',
                                       'metrics.test_macro_f1': 'Test Macro-F1'})
    x      = np.arange(len(plot_df))
    width  = 0.35
    fig, ax = plt.subplots(figsize=(max(8, len(plot_df)*2), 5))
    bars1 = ax.bar(x - width/2, plot_df['Val Macro-F1'],  width, label='Val Macro-F1',  color='steelblue')
    bars2 = ax.bar(x + width/2, plot_df['Test Macro-F1'], width, label='Test Macro-F1', color='darkorange')
    ax.set_xticks(x); ax.set_xticklabels(plot_df['Run'], rotation=20, ha='right')
    ax.set_ylabel('Macro-F1'); ax.set_ylim(0, 1)
    ax.set_title('Model Comparison — Val vs Test Macro-F1')
    ax.legend()
    for bar in bars1: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                               f'{bar.get_height():.3f}', ha='center', fontsize=8)
    for bar in bars2: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                               f'{bar.get_height():.3f}', ha='center', fontsize=8)
    plt.tight_layout(); plt.show()

## 11. Launch MLflow UI

In [ ]:
import subprocess, os

mlruns_abs = str(Path('../mlruns').resolve())
print('To view all runs in the MLflow UI, run this in your terminal:')
print(f'\n  mlflow ui --backend-store-uri {mlruns_abs} --port 5000')
print('\nThen open: http://127.0.0.1:5000')
print(f'\nExperiment : {EXPERIMENT_NAME}')
print(f'Total runs : {len(runs_df)}')

# Optionally launch in background from the notebook (comment out if not needed)
# subprocess.Popen(['mlflow', 'ui', '--backend-store-uri', mlruns_abs, '--port', '5000'])
# print('MLflow UI started in background.')